# What a program-only agent can establish about an unknown environmentCompanion notebook to our ARC Prize 2026 Paper Track writeup.It runs offline. The simulation comparison below is **computed live** when you runthis notebook; the figures measured against recorded real games are shipped as asmall JSON, because those traces are 2.8&nbsp;GB and Kaggle notebooks have no network.Every number here traces back to a document in the repository that records the runwhich produced it — including the numbers that contradict our earlier claims.

## SetupThe `arcagi3` library is pure Python and needs only `numpy`, which Kaggle already has.Attach the repository as a dataset, or run this notebook from inside a clone.

In [ ]:
import sys, json, pathlib# Find the library whether this runs from a clone or from an attached dataset.CANDIDATES = [    pathlib.Path("agi3"),    pathlib.Path("../agi3"),    *pathlib.Path("/kaggle/input").glob("*/agi3"),    *pathlib.Path("/kaggle/input").glob("*/*/agi3"),]for path in CANDIDATES:    if (path / "arcagi3").is_dir():        sys.path.insert(0, str(path))        print(f"library: {path.resolve()}")        breakelse:    raise SystemExit("could not find agi3/ — attach the repository as a dataset")DATA = next(    (p for p in [pathlib.Path("paper/data"), pathlib.Path("../paper/data"),                 *pathlib.Path("/kaggle/input").glob("*/paper/data"),                 *pathlib.Path("/kaggle/input").glob("*/*/paper/data")]     if p.is_dir()),    None,)print(f"measured results: {DATA}")

## 1. Simulation does not predict realityTwo arenas, the same policies. One is the quiet environment we wrote first; theother carries noise measured from recorded games — a HUD strip that advances onevery action, sprites that change shape as they move, and as a consequence a boardthat changes on essentially every action.This cell computes the table rather than printing a stored one.

In [ ]:
from arcagi3.agent import GreedyAgent, RandomAgentfrom arcagi3.budget import run_episodefrom arcagi3.explorer import ExplorerAgentfrom arcagi3.mock import MockEnvironmentfrom arcagi3.navigator import NavigatorAgentfrom arcagi3.noisy_mock import NoisyEnvironmentPOLICIES = {    "random": lambda: RandomAgent(seed=0),    "greedy": GreedyAgent,    "explorer": ExplorerAgent,    "navigator": NavigatorAgent,}ARENAS = {"quiet mock": MockEnvironment, "noise-calibrated": NoisyEnvironment}print(f"{'policy':<11}{'learns?':>9}" + "".join(f"{name:>20}" for name in ARENAS))print("-" * 60)for name, make_policy in POLICIES.items():    learns = "no" if name in ("random", "greedy") else "yes"    cells = []    for make_arena in ARENAS.values():        r = run_episode(make_policy(), make_arena(), max_actions=400)        cells.append(f"{'WIN' if r.won else 'lost'} {r.levels_completed}/{r.total_levels}")    print(f"{name:<11}{learns:>9}" + "".join(f"{c:>20}" for c in cells))

Three things follow.The noise destroys **only the methods that learn**. The random and hard-codedpolicies score identically in both arenas, because neither holds a model for thenoise to corrupt.On a board carrying the noise real boards carry, **random play beats both of ourdeliberate policies**, two levels to none.And results in the quiet arena carry **no information** about the noisy one: threepolicies clear every level there, and their fates then diverge completely.

## 2. What probing costs, measured on real boardsThese come from replaying 500 recorded runs — 25 official games, 20 passes each —and checking the mapping the learner recovers against the action labels therecordings carry.

In [ ]:
measured = json.loads((DATA / "control-learning-measured.json").read_text())curve = measured["learning_curve"]print(measured["_provenance"]["source"])print()print(f"{'observations':>13}{'mappings':>10}{'correct':>9}{'accuracy':>10}")print("-" * 42)for row in curve:    print(f"{row['observations']:>13}{row['mappings']:>10}{row['correct']:>9}"          f"{row['accuracy']:>9.0%}")

In [ ]:
import matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(7, 4))ax.plot([r["observations"] for r in curve], [r["accuracy"] * 100 for r in curve],        marker="o", color="#2b6cb0")ax.set_xscale("log")ax.set_xlabel("observations fed to the learner (log scale)")ax.set_ylabel("mapping accuracy (%)")ax.set_title("Learning the controls: accuracy against evidence")ax.grid(alpha=0.3)ax.set_ylim(0, 100)plt.tight_layout()plt.show()

Nothing is learnable below about ten observations, accuracy climbs to roughly 80%by eighty, and then flattens.A separate measurement matters more in play: a mapping *correct about twodirections* costs a median of **12 observations**. Those answer differentquestions, and conflating them — as we did — makes starting look far moreexpensive than it is. Two correct directions are enough to begin.

## 3. Is the learner's confidence honest?It reports a confidence alongside each mapping. If that number means anything,accuracy should rise with it.

In [ ]:
bands = measured["confidence_calibration"]print(f"{'confidence':>14}{'mappings':>10}{'correct':>9}{'accuracy':>10}")print("-" * 43)for row in bands:    label = f"{row['confidence_from']:.1f}-{row['confidence_to']:.1f}"    print(f"{label:>14}{row['mappings']:>10}{row['correct']:>9}{row['accuracy']:>9.0%}")fig, ax = plt.subplots(figsize=(7, 4))labels = [f"{r['confidence_from']:.1f}-{r['confidence_to']:.1f}" for r in bands]ax.bar(labels, [r["accuracy"] * 100 for r in bands], color="#2b6cb0")ax.set_xlabel("reported confidence")ax.set_ylabel("observed accuracy (%)")ax.set_title("Confidence is informative, with a cliff at 0.6")ax.set_ylim(0, 100)ax.grid(axis="y", alpha=0.3)plt.tight_layout()plt.show()

There is a sharp threshold at **0.6**: below it the mapping is 36% accurate, aboveit 84–85%. Waiting for higher confidence buys no further accuracy while costingactions.It is informative but not a guarantee — even the top band is wrong 15% of the time.

## 4. Four times the simulator convinced us| # | Looked settled | Then failed | Mechanism ||---|---|---|---|| 1 | object matching, 88% in our mock | **0.4%** on 2,276 real transitions | an animating sprite does not match itself between frames || 2 | "the board changed" as a click signal | **worthless** — 91–100% of real clicks change the board | a signal present almost always separates nothing || 3 | our best walking policy, 3/3 levels | **0/3** once the board is noisy | the centre of mass is dragged sideways by animation || 4 | cell alignment, 3/3 on the *calibrated* mock | **55%** on real boards, against the centroid's 79% | alignment discards an observation whenever nothing lines up |The fourth is the one that matters most. The arena it passed was itself calibratedfrom statistics measured over 500 real runs, and it still selected the worse design.Matching measured statistics is not sufficient, because the statistics one choosesto match come from the same understanding that shaped the method being tested.We later recovered 22 of those 24 points by falling back to the centre of mass whenalignment finds nothing — so the failure was real, but its cause was discardedevidence rather than a conflict of principle.

## 5. Which noise actually mattersSwitching each property on and off separately, against our own walking game:| noise applied | policy result ||---|---|| animating the controlled object only | **clears 0 of 3 levels** || animating every small object | clears 3 of 3 || animating only the goal | clears 3 of 3 || HUD alone, nothing animated | clears 3 of 3 || controlled object animated, no HUD | **clears 0 of 3** |More noise made the test **easier**. It is not noise that defeats learning, butnoise on the signal being learned from — which is a statement about learningagents rather than about ARC.

## 6. Three checksThese would have caught all four of our failures, and they need no ARC-specificknowledge:1. **Test against data you did not generate.** Recordings, traces, a live game.2. **Measure the base rate of whatever signal you learn from.** A signal present   95% of the time separates nothing, however reasonable it looks.3. **Measure component accuracy and playing ability separately.** Ours diverged   completely: 79% mapping accuracy, zero levels cleared.The noise itself is packaged as a wrapper that takes any environment returning`FrameData`, so the same check runs against another team's agent unchanged:```pythonfrom arcagi3.noise import NoiseWrappernoisy = NoiseWrapper(YourEnvironment(), sprite_colour=<the object you steer>)```Naming the object under control is what makes it a hard test — the table insection 5 is why.

In [ ]:
# The wrapper takes an environment it has never seen.from arcagi3.click_mock import ClickEnvironmentfrom arcagi3.noise import NoiseWrapperframe = NoiseWrapper(ClickEnvironment()).reset()print(f"wrapped a different game: board is {len(frame.frame[-1])}x{len(frame.frame[-1][0])}")

## Reproducing all of it```bashgit clone https://github.com/Panus15/arc-prize-2026cd arc-prize-2026/agi3 && ./setup.sh.venv/bin/pytest                                      # 196 testsPYTHONPATH=. .venv/bin/python scripts/simulation_gap.py   # the table in section 1PYTHONPATH=. .venv/bin/python scripts/validate_control.py # sections 2 and 3```The repository is MIT-0. Each figure above has a document in `docs/` recording therun behind it.